# Mental Health Support Chatbot (Fine-Tuned)


In [6]:
# Step 1: Import Libraries
 

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

## Step 2: Load Dataset

In [11]:
dataset = load_dataset("empathetic_dialogues")

# Use only small portion for faster training
train_data = dataset['train'].select(range(1000))


## Step 3: Load Model and Tokenizer

In [12]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 76/76 [00:

## Step 4: Prepare Dataset

In [14]:
def preprocess(example):
    
    text = (
        "User: " + example['utterance'] +
        "\nBot: " + example['context']
    )
    
    return tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128
    )

tokenized_dataset = train_data.map(preprocess)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 1078.12 examples/s]


## Step 5: Data Collator

In [15]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Step 6: Training Arguments

In [18]:
training_args = TrainingArguments(
    output_dir="./mental_health_chatbot",
    
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100
)

## Step 7: Trainer

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

## Step 8: Train Model

In [20]:
trainer.train()

# Save trained model
trainer.save_model("./mental_health_chatbot")

print("Model training completed!")


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.355856
200,2.965497
300,2.924058
400,2.895095
500,2.771765


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]

Model training completed!


## Step 9: Chatbot Testing Function

In [21]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="./mental_health_chatbot",
    tokenizer=tokenizer
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1479.14it/s]


In [24]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="./mental_health_chatbot",
    tokenizer=tokenizer
)

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    prompt = f"User: {user_input}\nBot:"

    response = chatbot(
        prompt,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7
    )

    print("\nChatbot:")
    print(response[0]['generated_text'])

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3129.22it/s]
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot:
User: i am not happy
Bot: furious at the situation
Bot: furious at the situation
Bot: furious at the situation
Bot: furious at the situation
Bot: furious at


Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot:
User: 
Bot: confident_comma_ but not confident_comma_
Bot: confident_comma_ but not confident_comma_
Bot:


Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Chatbot:
User: 
Bot: anxious 
Bot: anxious
Bot: anxious
Bot: anxious
Bot: anxious
Bot: anxious
Bot: anxious
Bot: anxious


KeyboardInterrupt: Interrupted by user

In [23]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="./mental_health_chatbot",
    tokenizer=tokenizer
)

user_input = "I feel stressed these days"

prompt = f"User: {user_input}\nBot:"

response = chatbot(
    prompt,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7
)

print(response[0]['generated_text'])

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3257.71it/s]
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: I feel stressed these days
Bot: anxious about a new job
Bot: anxious about new job
Bot: anxious about new job
Bot: anxious about new job
Bot: anxious
